```
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
```

# RMI Sample Queries: Urban Planner (GA)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Urban_Planner_Samples.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Frmi_sample_queries%2Fnotebooks%2FUrban_Planner_Samples.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Urban_Planner_Samples.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Urban_Planner_Samples.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>



This notebook contains sample queries for the **Urban Planner** persona, specifically for the **GA** stage.

## 1. Setup

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()

project_id = 'your-project-id' #@param {type:"string"}

### Writable Dataset

Several queries in this notebook (e.g., those creating Materialized Views, Models, or Views) require a **writable dataset** within your own project. 
**Note**: The source `boston_oct_2025_sample_data` dataset is a read-only subscription and cannot be used to store new resources.

Run the cell below to create a new dataset (e.g., `rmi_analysis`) in your project if you haven't already.

**Important**: When running queries that create new BigQuery resources (e.g., tables, views, models) outside of these `%%bigquery` magic cells, remember to manually prepend the job ID with `msqlfactory--` for proper tracking. For example: `bq query --job_id=msqlfactory--your-descriptive-job-name ...`

In [ ]:
dataset_id = "rmi_analysis" #@param {type:"string"}
! bq --location=US mk --dataset {project_id}:{dataset_id}

## GA (General Availability)

### up1_corridor_trend.sql
**Business Question**: What has been the week-over-week trend in the average delay ratio for a specific corridor?
Use Case: Enables long-term performance monitoring of critical transportation infrastructure, helping planners identify if congestion is worsening or improving over time.
Product Stage: GA
Estimated Bytes Processed: ~150 MB

In [ ]:
%%bigquery --project {project_id} df_up1_corridor_trend
/*
  ANALYTICAL PATTERN: Weekly Trend Aggregation
  This query truncates timestamps to the week level to smooth out day-to-day 
  fluctuations, focusing on the macro traffic behavior of a critical route.
*/

WITH weekly_trends AS (
  SELECT
    selected_route_id,
    -- Truncate to the start of the week for consistent aggregation
    TIMESTAMP_TRUNC(record_time, WEEK) AS week,
    -- Calculate average delay (Actual / Free-flow baseline)
    AVG(SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds)) AS avg_delay_ratio
  FROM `boston_oct_2025_sample_data.historical_travel_time`
  -- Filter for a specific corridor of interest (e.g., Storrow Drive)
  WHERE selected_route_id = 'route-4202493217'
    AND record_time BETWEEN '2025-10-01' AND '2025-11-01'
  GROUP BY 1, 2
)
SELECT
  selected_route_id,
  -- Format for readable year-week reporting
  FORMAT_TIMESTAMP("%Y-%W", week) AS year_week,
  ROUND(avg_delay_ratio, 3) AS avg_delay_ratio
FROM weekly_trends
ORDER BY week;


### up2_impact_analysis.sql
**Business Question**: Has the average travel time on routes passing through a recent construction zone improved since the project's completion date?
Use Case: Provides empirical evidence of infrastructure project success, validating whether road improvements (like new lanes or signals) actually reduced congestion.
Product Stage: GA
Estimated Bytes Processed: ~150 MB

In [ ]:
%%bigquery --project {project_id} df_up2_impact_analysis
/*
  ANALYTICAL PATTERN: Spatial & Milestone Join
  This query uses a DECLARE statement for the study area geometry to ensure 
  BigQuery treats the polygon as a constant, enabling efficient spatial 
  indexing during the ST_INTERSECTS join. It then segments the data based 
  on a chronological project milestone.
*/

-- Study Area: Downtown Boston Intersection
DECLARE study_area GEOGRAPHY DEFAULT ST_GEOGFROMTEXT('POLYGON((-71.06 42.35, -71.05 42.35, -71.05 42.34, -71.06 42.34, -71.06 42.35))');
-- Project Milestone: Date when construction was completed
DECLARE completion_date DATE DEFAULT '2025-10-15';

WITH impact_data AS (
  SELECT
    -- Split records into 'Before' and 'After' buckets
    record_time >= completion_date AS is_after_completion,
    SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) AS delay_ratio
  FROM `boston_oct_2025_sample_data.historical_travel_time`
  -- Filter for routes that physically pass through the study zone
  WHERE ST_INTERSECTS(route_geometry, study_area)
    AND record_time BETWEEN '2025-10-01' AND '2025-11-01'
)
SELECT
  is_after_completion,
  ROUND(AVG(delay_ratio), 3) AS avg_delay_ratio,
  COUNT(*) as sample_count
FROM impact_data
GROUP BY 1
ORDER BY 1;


### up3_monitoring_density.sql
**Business Question**: Which geographic areas show the highest concentration of RMI route monitoring?
Use Case: Helps planners identify "blind spots" in their monitoring network or confirm that critical urban zones are sufficiently covered by RMI probes.
Product Stage: GA
Estimated Bytes Processed: ~150 MB

In [ ]:
%%bigquery --project {project_id} df_up3_monitoring_density
/*
  ANALYTICAL PATTERN: Spatial Grid Aggregation
  This query maps the RMI monitoring footprint by calculating route centroids 
  and grouping them into a ~1.1km grid (3 decimal places). This provides a 
  coarse-grained view of network density without high computational overhead.
*/

WITH route_centroids AS (
  SELECT 
    selected_route_id,
    -- Use the centroid to represent the general location of the route polyline
    ST_CENTROID(route_geometry) as centroid
  FROM `boston_oct_2025_sample_data.historical_travel_time`
  WHERE record_time BETWEEN '2025-10-01' AND '2025-11-01'
)
SELECT
  -- Grid the coordinates to a precision of ~1.1km
  ROUND(ST_Y(centroid), 3) AS lat_grid,
  ROUND(ST_X(centroid), 3) AS lon_grid,
  -- Count unique route definitions in this grid cell
  COUNT(DISTINCT selected_route_id) AS unique_routes_monitored,
  -- Count total traffic samples captured in this grid cell
  COUNT(*) AS total_samples_collected
FROM route_centroids
GROUP BY 1, 2
ORDER BY unique_routes_monitored DESC
LIMIT 20;


### up4_weekend_vs_weekday.sql
**Business Question**: How does average travel time in the afternoon (2-5 PM) differ between weekdays and weekends?
Use Case: Informs urban policy decisions like congestion pricing or off-peak transit scheduling by highlighting when road demand is most elastic.
Product Stage: GA
Estimated Bytes Processed: ~150 MB

In [ ]:
%%bigquery --project {project_id} df_up4_weekend_vs_weekday
/*
  ANALYTICAL PATTERN: Day-Type Segmentation
  This query uses EXTRACT(DAYOFWEEK...) to categorize records into binary 
  'Weekday' or 'Weekend' buckets. It combines this with a peak-window filter 
  to provide a clean comparison of temporal demand shifts.
*/

WITH afternoon_stats AS (
  SELECT
    -- Day segmentation: 1 = Sunday, 7 = Saturday
    CASE 
      WHEN EXTRACT(DAYOFWEEK FROM DATETIME(record_time, 'America/New_York')) IN (1, 7) THEN 'Weekend'
      ELSE 'Weekday'
    END AS day_type,
    duration_in_seconds,
    static_duration_in_seconds
  FROM `boston_oct_2025_sample_data.historical_travel_time`
  WHERE record_time BETWEEN '2025-10-01' AND '2025-11-01'
    -- Afternoon period: 2 PM to 5 PM Local Time (Boston)
    AND EXTRACT(HOUR FROM DATETIME(record_time, 'America/New_York')) BETWEEN 14 AND 17
)
SELECT
  day_type,
  -- Calculate Average Delay Index (Actual / Ideal)
  ROUND(AVG(SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds)), 3) AS avg_delay_ratio,
  ROUND(AVG(duration_in_seconds), 2) AS avg_duration_seconds,
  COUNT(*) as sample_count
FROM afternoon_stats
GROUP BY 1
ORDER BY avg_delay_ratio DESC;


### up5_geofenced_congestion.sql
**Business Question**: Within a specific downtown polygon, which routes are currently seeing travel times more than 50% above their static baseline?
Use Case: Enables targeted traffic management and demand-response strategies within high-density zones or during special events.
Product Stage: GA
Estimated Bytes Processed: ~150 MB

In [ ]:
%%bigquery --project {project_id} df_up5_geofenced_congestion
/*
  ANALYTICAL PATTERN: Spatial Geofencing
  This query uses a DECLARE statement for the downtown polygon to ensure 
  BigQuery treats the study area as a constant, enabling efficient spatial 
  indexing during the ST_INTERSECTS join. It identifies routes that are 
  physically impacted by a specific urban zone.
*/

-- Study Area: Downtown Boston Geofence
DECLARE downtown_zone GEOGRAPHY DEFAULT ST_GEOGFROMTEXT('POLYGON((-71.066 42.358, -71.052 42.358, -71.052 42.348, -71.066 42.348, -71.066 42.358))');

WITH intersecting_routes AS (
  SELECT
    h.selected_route_id,
    h.display_name,
    SAFE_DIVIDE(h.duration_in_seconds, h.static_duration_in_seconds) AS delay_ratio
  FROM `boston_oct_2025_sample_data.historical_travel_time` h
  WHERE ST_INTERSECTS(h.route_geometry, downtown_zone)
    AND h.record_time BETWEEN '2025-10-01' AND '2025-11-01'
    -- Quality filter: Exclude non-continuous geometries
    AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
)
SELECT
  selected_route_id,
  display_name,
  ROUND(AVG(delay_ratio), 3) AS avg_delay_ratio,
  COUNT(*) as sample_count
FROM intersecting_routes
GROUP BY 1, 2
-- Threshold: Filter for routes that are at least 1.5x slower than free-flow
HAVING avg_delay_ratio > 1.5
ORDER BY avg_delay_ratio DESC;
